# Treinar Tokenizer BPE para o Tutor de Teoria Musical

Notebook de treino do tokenizer BPE usado pelo corpus estruturado.


In [14]:
# Instale apenas se necessário.
# %pip install "tokenizers>=0.15.0"

import json
import re
from collections import Counter
from pathlib import Path

from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.models import BPE
from tokenizers.normalizers import NFKC
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer

## 1. Configuração

In [15]:
DATA_DIR = Path("data")
TOKENIZER_DIR = Path("tokenizer")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / "teoria_musical_treino_200k.txt"
VAL_FILE = DATA_DIR / "teoria_musical_validacao_20k.txt"
TEST_FILE = DATA_DIR / "teoria_musical_teste_20k.txt"

TOKENIZER_PATH = TOKENIZER_DIR / "tokenizer.json"
TOKENIZER_CONFIG_PATH = TOKENIZER_DIR / "tokenizer_config.json"

VOCAB_SIZE = 8_000
MIN_FREQUENCY = 2

# tokens especiais + as tags de abertura/fechamento usadas no corpus
SPECIAL_TOKENS = [
    "<pad>", "<bos>", "<eos>", "<unk>",
    "<registro>", "</registro>",
    "<conceito>", "</conceito>",
    "<conteudo>", "</conteudo>",
    "<pergunta>", "</pergunta>",
    "<resposta>", "</resposta>",
    "<exercicio>", "</exercicio>",
    "<analise>", "</analise>",
]

for path in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path.resolve()}")

print("Arquivo de treino:", TRAIN_FILE)
print("Arquivo de validação:", VAL_FILE)
print("Arquivo de teste:", TEST_FILE)
print("Vocabulário solicitado:", VOCAB_SIZE)
print("Frequência mínima:", MIN_FREQUENCY)

Arquivo de treino: data\teoria_musical_treino_200k.txt
Arquivo de validação: data\teoria_musical_validacao_20k.txt
Arquivo de teste: data\teoria_musical_teste_20k.txt
Vocabulário solicitado: 8000
Frequência mínima: 2


## 2. Inspecionar os arquivos

Só o treino entra no ajuste do BPE. Validação e teste ficam separados e servem pra conferir cobertura mais na frente.

In [16]:
def count_words(text: str) -> int:
    return len(re.findall(r"\S+", text))


corpus_texts = {
    "train": TRAIN_FILE.read_text(encoding="utf-8"),
    "val": VAL_FILE.read_text(encoding="utf-8"),
    "test": TEST_FILE.read_text(encoding="utf-8"),
}

for split_name, text in corpus_texts.items():
    if not text.strip():
        raise ValueError(f"O conjunto {split_name} está vazio.")

    print(f"{split_name:>5}: {count_words(text):,} palavras | {len(text):,} caracteres")

train: 200,000 palavras | 1,436,803 caracteres
  val: 20,000 palavras | 144,255 caracteres
 test: 20,000 palavras | 143,513 caracteres


## 3. As tags do corpus batem com o esperado?

In [17]:
TAG_PATTERN = re.compile(r"</?[^>\s]+(?:\s+[^>]*)?>")

tags_by_split = {
    split_name: Counter(TAG_PATTERN.findall(text))
    for split_name, text in corpus_texts.items()
}

for split_name, tag_counts in tags_by_split.items():
    print(f"\nTags em {split_name}:")
    for tag, count in sorted(tag_counts.items()):
        print(f"{tag:20s} {count:,}")

known_tags = set(SPECIAL_TOKENS)
corpus_tags = set().union(*tags_by_split.values())

# <registro id="..."> carrega atributo, então não bate exatamente com o token
# especial -- só </registro> bate, e é o que interessa pro tokenizer.
unknown_exact_tags = {
    tag for tag in corpus_tags
    if tag not in known_tags and not tag.lower().startswith("<registro ")
}

if unknown_exact_tags:
    print("\nAviso: tags não cadastradas como especiais:")
    for tag in sorted(unknown_exact_tags):
        print("-", tag)
else:
    print("\nNão foram encontradas tags inesperadas.")


Tags em train:
</analise>           444
</conceito>          445
</conteudo>          445
</exercicio>         444
</pergunta>          445
</registro>          1,778
</resposta>          445
<analise>            444
<conceito>           445
<conteudo>           445
<exercicio>          445
<pergunta>           445
<registro id="treino-A-100"> 1
<registro id="treino-A-1000"> 1
<registro id="treino-A-1004"> 1
<registro id="treino-A-1008"> 1
<registro id="treino-A-1012"> 1
<registro id="treino-A-1016"> 1
<registro id="treino-A-1020"> 1
<registro id="treino-A-1024"> 1
<registro id="treino-A-1028"> 1
<registro id="treino-A-1032"> 1
<registro id="treino-A-1036"> 1
<registro id="treino-A-104"> 1
<registro id="treino-A-1040"> 1
<registro id="treino-A-1044"> 1
<registro id="treino-A-1048"> 1
<registro id="treino-A-1052"> 1
<registro id="treino-A-1056"> 1
<registro id="treino-A-1060"> 1
<registro id="treino-A-1064"> 1
<registro id="treino-A-1068"> 1
<registro id="treino-A-1072"> 1
<registro id

## 4. Treinar o tokenizer BPE

O `ByteLevel` preserva acento, quebra de linha, cifra, sustenido, bemol e os símbolos estruturais do corpus. Em cima disso, o BPE aprende as combinações de bytes mais frequentes.

In [18]:
tokenizer = Tokenizer(BPE(unk_token="<unk>", fuse_unk=True))

tokenizer.normalizer = NFKC()
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False, use_regex=True)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS,
    initial_alphabet=ByteLevel.alphabet(),
    show_progress=True,
)

# só o arquivo de treino participa do ajuste do BPE
tokenizer.train(files=[str(TRAIN_FILE)], trainer=trainer)
tokenizer.save(str(TOKENIZER_PATH))

print("Tokenizer salvo em:", TOKENIZER_PATH.resolve())
print("Vocabulário final:", tokenizer.get_vocab_size())

Tokenizer salvo em: C:\Users\jgmda\Documents\GitHub\Doutorado\DeepLearningLLMs\TrabalhoFinal\tokenizer\tokenizer.json
Vocabulário final: 1730


## 5. Salvar configuração auxiliar

In [19]:
tokenizer_config = {
    "tokenizer_type": "BPE",
    "pre_tokenizer": "ByteLevel",
    "normalizer": "NFKC",
    "vocab_size_requested": VOCAB_SIZE,
    "vocab_size_actual": tokenizer.get_vocab_size(),
    "min_frequency": MIN_FREQUENCY,
    "training_file": TRAIN_FILE.name,
    "validation_file": VAL_FILE.name,
    "test_file": TEST_FILE.name,
    "special_tokens": SPECIAL_TOKENS,
}

TOKENIZER_CONFIG_PATH.write_text(
    json.dumps(tokenizer_config, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Configuração salva em:", TOKENIZER_CONFIG_PATH.resolve())

Configuração salva em: C:\Users\jgmda\Documents\GitHub\Doutorado\DeepLearningLLMs\TrabalhoFinal\tokenizer\tokenizer_config.json


## 6. IDs dos tokens especiais

In [20]:
special_token_ids = {}

for token in SPECIAL_TOKENS:
    token_id = tokenizer.token_to_id(token)
    special_token_ids[token] = token_id
    print(f"{token:18s} -> {token_id}")

missing = [token for token, token_id in special_token_ids.items() if token_id is None]
if missing:
    raise ValueError("Tokens especiais ausentes: " + ", ".join(missing))

if len(set(special_token_ids.values())) != len(SPECIAL_TOKENS):
    raise ValueError("Dois ou mais tokens especiais receberam o mesmo ID.")

<pad>              -> 0
<bos>              -> 1
<eos>              -> 2
<unk>              -> 3
<registro>         -> 4
</registro>        -> 5
<conceito>         -> 6
</conceito>        -> 7
<conteudo>         -> 8
</conteudo>        -> 9
<pergunta>         -> 10
</pergunta>        -> 11
<resposta>         -> 12
</resposta>        -> 13
<exercicio>        -> 14
</exercicio>       -> 15
<analise>          -> 16
</analise>         -> 17


## 7. Teste de tokenização estruturada

In [21]:
example = '''<registro>
<pergunta>O que é uma escala maior?</pergunta>
<resposta>Uma escala maior segue o padrão tom, tom, semitom, tom, tom, tom, semitom. Em Dó maior, as notas são Dó, Ré, Mi, Fá, Sol, Lá e Si.</resposta>
</registro>'''

encoded = tokenizer.encode(example)
decoded = tokenizer.decode(encoded.ids, skip_special_tokens=False)

print("Texto original:")
print(example)

print("\nPrimeiros IDs:")
print(encoded.ids[:100])

print("\nPrimeiros tokens:")
print(encoded.tokens[:100])

print("\nTexto decodificado:")
print(decoded)

Texto original:
<registro>
<pergunta>O que é uma escala maior?</pergunta>
<resposta>Uma escala maior segue o padrão tom, tom, semitom, tom, tom, tom, semitom. Em Dó maior, as notas são Dó, Ré, Mi, Fá, Sol, Lá e Si.</resposta>
</registro>

Primeiros IDs:
[4, 216, 10, 64, 383, 757, 357, 1125, 486, 48, 11, 216, 12, 70, 296, 1125, 486, 333, 88, 751, 290, 318, 85, 99, 280, 437, 94, 29, 437, 94, 29, 803, 298, 94, 29, 437, 94, 29, 437, 94, 29, 437, 94, 29, 803, 298, 94, 31, 1030, 1103, 486, 29, 404, 393, 292, 280, 1103, 29, 1172, 29, 1137, 29, 1136, 29, 1102, 29, 1100, 274, 1098, 31, 13, 216, 5]

Primeiros tokens:
['<registro>', 'Ċ', '<pergunta>', 'O', 'Ġque', 'ĠÃ©', 'Ġuma', 'Ġescala', 'Ġmaior', '?', '</pergunta>', 'Ċ', '<resposta>', 'U', 'ma', 'Ġescala', 'Ġmaior', 'Ġse', 'g', 'ue', 'Ġo', 'Ġpa', 'd', 'r', 'Ã£o', 'Ġto', 'm', ',', 'Ġto', 'm', ',', 'Ġsemi', 'to', 'm', ',', 'Ġto', 'm', ',', 'Ġto', 'm', ',', 'Ġto', 'm', ',', 'Ġsemi', 'to', 'm', '.', 'ĠEm', 'ĠDÃ³', 'Ġmaior', ',', 'Ġas', 'Ġnotas', '